# Portefeuille des élus du Congrès — spécification mathématique
**Reconstruction, classement, stratégie de copie et ajustement au marché**

Notebook calqué sur `NOTE_MATHS_PORTEFEUILLE_MEMBRE`. Réutilise le pipeline de données du `05`,
mais avec la **combinaison inter-membres corrigée** (poids qui dérivent, buy-and-hold) et
l'**ajustement par le marché** (β, α, ratio d'appréciation).

Notations : $i$ = ticker, $t$ = jour de bourse, $P_i(t)$ = cours de clôture.

## 0. Données (nettoyage transactionnel fait en amont S1/S2)
On charge la table canonique source-primaire, on aligne les noms de colonnes, on restreint la fenêtre,
puis on construit le panel de prix (cache `prices_v2`) avec le garde-fou anti-corruption Yahoo.

In [1]:
%matplotlib inline
import pandas as pd, numpy as np, glob, os
import matplotlib.pyplot as plt

BASE = "/Users/lemairealice/Downloads/Jupiter/00. S3S4 en cours"
PRICES = os.path.join(BASE, "cache", "prices_v2")

In [2]:
# Table PROPRE canonique S1/S2 (source-primaire) -> noms alignés sur la mécanique du 05.
CLEAN = "/Users/lemairealice/Downloads/Jupiter/00_S1S2_donnees/data/clean/transactions_backtest_2014_2026.csv"
df = pd.read_csv(CLEAN, low_memory=False)
df = (df.drop(columns=["ticker"])
        .rename(columns={"member_name": "name", "ticker_yahoo": "ticker",
                         "direction": "op", "transaction_date": "traded",
                         "amount_midpoint": "size_usd"}))
n0 = len(df)
df["traded"] = pd.to_datetime(df["traded"], errors="coerce")
df = df[df["traded"].dt.year.between(2013, 2026)].reset_index(drop=True)
df = df[["bioguide_id", "name", "ticker", "op", "traded", "size_usd"]]
print(f"{n0} -> {len(df)} trades  |  {df['bioguide_id'].nunique()} membres  |  {df['ticker'].nunique()} tickers")

134464 -> 134429 trades  |  372 membres  |  4618 tickers


In [3]:
# SPY = benchmark ; son index = calendrier maître
spy = (pd.read_csv(os.path.join(PRICES, "SPY.csv"), parse_dates=["Date"])
         .set_index("Date")["close"].sort_index())
cal = spy.index
r_spy = spy.pct_change()          # rendement quotidien du marché

In [4]:
# Télécharge les prix ABSENTS du cache (resumable ; cache complet => 0 => hors-ligne)
import yfinance as yf, time, logging
logging.getLogger("yfinance").setLevel(logging.CRITICAL)
os.makedirs(PRICES, exist_ok=True)
have0  = {os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(PRICES, "*.csv"))}
fail_f = os.path.join(PRICES, "failed_v2.csv")
failed = set(pd.read_csv(fail_f)["ticker"]) if os.path.exists(fail_f) else set()
todo   = [t for t in dict.fromkeys(df["ticker"]) if isinstance(t, str) and t not in have0 and t not in failed]
print(f"cache: {len(have0)} tickers | délistés/échecs connus sautés: {len(set(df['ticker']) & failed)} | à télécharger: {len(todo)}")
new_fail = []
for i in range(0, len(todo), 60):
    chunk = todo[i:i+60]
    try:
        data = yf.download(chunk, start="2012-01-01", end="2026-07-03", auto_adjust=True,
                           progress=False, threads=True, group_by="ticker")
    except Exception as e:
        print("  batch KO:", str(e)[:60]); data = None
    for t in chunk:
        s = None
        try:
            if data is not None and isinstance(data.columns, pd.MultiIndex):
                if t in data.columns.get_level_values(0):
                    s = data[t]["Close"].dropna()
            elif data is not None and "Close" in getattr(data, "columns", []):
                s = data["Close"].dropna()
        except Exception:
            s = None
        if s is None or len(s) < 20:
            new_fail.append(t)
        else:
            s.rename("close").to_csv(os.path.join(PRICES, t + ".csv"))
    time.sleep(0.4)
if new_fail:
    pd.DataFrame({"ticker": new_fail, "classe": "inattendu"}).to_csv(fail_f, mode="a", header=False, index=False)
print(f"téléchargés OK: {len(todo) - len(new_fail)} | nouveaux échecs: {len(new_fail)}")

cache: 3351 tickers | délistés/échecs connus sautés: 1268 | à télécharger: 0
téléchargés OK: 0 | nouveaux échecs: 0


In [5]:
# Charge les prix utiles, alignés sur le calendrier ; garde-fou anti-corruption Yahoo
have = {os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(PRICES, "*.csv"))}
need = sorted(set(df["ticker"]) & have)
missing = sorted(set(df["ticker"]) - have)
prices = {}
for tk in need:
    s = (pd.read_csv(os.path.join(PRICES, tk + ".csv"), parse_dates=["Date"])
           .set_index("Date")["close"].sort_index())
    prices[tk] = s.reindex(cal).ffill()
corrompus = [tk for tk in need if (len(prices[tk].dropna()) < 2
             or prices[tk].dropna().min() < 0.10
             or prices[tk].dropna().pct_change().abs().max() > 3.0)]   # <0,10$ ou saut >300%/j
for tk in corrompus:
    del prices[tk]
need = [tk for tk in need if tk not in set(corrompus)]
n_drop = df[~df["ticker"].isin(need)].shape[0]
df = df[df["ticker"].isin(need)].reset_index(drop=True)
print(f"tickers avec prix : {len(need)} | sans prix : {len(missing)} | corrompus exclus : {len(corrompus)} | trades jetés : {n_drop}  => {len(df)} exploitables")

tickers avec prix : 3289 | sans prix : 1268 | corrompus exclus : 61 | trades jetés : 16400  => 118029 exploitables


## 1. Reconstruction du portefeuille d'un membre
- **Achat** $\to$ parts, au 1er jour coté $\ge$ date de transaction : $q = \dfrac{\text{montant}}{P_i(\tau)},\ \tau=\min\{t\ge \texttt{traded}\}$.
- **Vente** $\to$ retire des parts (short interdit) : $q^- = \min(\text{montant}/P_i,\ \text{parts détenues})$.
- **Détentions cumulées** : $n_i(t) = \sum_{\text{achats}\le t} q - \sum_{\text{ventes}\le t} q^-$.
- **Valeur** : $V(t) = \sum_i n_i(t)\,P_i(t)$.

In [6]:
def eff_price(tk, d):
    # prix du 1er jour de bourse >= d (on transige au marché)
    s = prices[tk]; pos = s.index.searchsorted(d)
    if pos >= len(s): return None, None
    p = s.iloc[pos]
    if not np.isfinite(p) or p <= 0: return None, None
    return s.index[pos], p

def member_shares(g):
    # g = trades d'un membre  ->  N (parts détenues par jour, colonnes = tickers)
    cols = {}
    for tk, gt in g.groupby("ticker"):
        h, events = 0.0, {}
        for _, row in gt.sort_values("traded").iterrows():
            dd, p = eff_price(tk, row["traded"])
            if dd is None:
                continue
            qty = row["size_usd"] / p
            h = h + qty if row["op"] == "buy" else h - min(qty, h)   # pas de short
            events[dd] = h
        if events:
            cols[tk] = pd.Series(events).sort_index().reindex(cal).ffill().fillna(0.0)
    return pd.DataFrame(cols) if cols else None

In [7]:
# Démo : un membre -> parts détenues n_i(t) (colonnes = tickers) et valeur V(t)
DEMO = "John Fetterman"
Nm = member_shares(df[df["name"] == DEMO])
V = (Nm * pd.DataFrame({tk: prices[tk] for tk in Nm.columns})).sum(axis=1)
print(f"{DEMO} : {Nm.shape[1]} tickers ; V(dernier jour) = {V.iloc[-1]:,.0f} $")
Nm[Nm.sum(axis=1) > 0].iloc[::max(1, (Nm.sum(axis=1) > 0).sum() // 8)].round(1)

John Fetterman : 7 tickers ; V(dernier jour) = 106,346 $


,AMZN,ERIE,GOOG,JPM,MSFT,MU,T
Date,,,,,,,
2025-05-15,0.0,0.0,48.6,0.0,0.0,0.0,0.0
2025-07-08,0.0,0.0,48.6,30.3,0.0,0.0,0.0
2025-08-26,0.0,0.0,48.6,30.3,0.0,0.0,0.0
2025-10-15,0.0,0.0,48.6,30.3,0.0,0.0,0.0
2025-12-04,0.0,0.0,48.6,30.3,0.0,0.0,0.0
2026-01-27,0.0,0.0,48.6,30.3,0.0,0.0,0.0
2026-03-18,0.0,0.0,48.6,30.3,0.0,0.0,0.0
2026-05-07,39.8,65.9,77.9,30.3,44.8,24.9,0.0
2026-06-29,39.8,65.9,77.9,30.3,44.8,24.9,0.0


## 2. Rendement d'un membre (*time-weighted*)
On valorise les parts de la **veille** $\Rightarrow$ un apport n'est jamais un rendement :
$$r_P(t) = \frac{\sum_i n_i(t-1)P_i(t)}{\sum_i n_i(t-1)P_i(t-1)} - 1$$
- Jour sans position : $r_P(t)=0$ (système ouvert). Garde-fou : $r_P \leftarrow \mathrm{clip}(r_P,-0.5,+0.5)$ (winsor ±50 %/j).
- $\mathrm{NAV}(t)=\prod_{s\le t}(1+r_P(s))$, $\mathrm{vol}=\sigma(r_P)\sqrt{252}$, $\mathrm{Sharpe}=\frac{\overline{r_P}}{\sigma(r_P)}\sqrt{252}$, maxDD.

In [8]:
def daily_return(N):
    P = pd.DataFrame({tk: prices[tk] for tk in N.columns})
    Nsh = N.shift(1)                                    # parts de la veille
    num = (Nsh * P).sum(axis=1)                         # Σ n_i(t-1) P_i(t)
    den = (Nsh * P.shift(1)).sum(axis=1)                # Σ n_i(t-1) P_i(t-1)
    r = pd.Series(np.where(den > 0, num / den - 1.0, 0.0), index=N.index)
    r = r.clip(-0.5, 0.5)                               # winsorisation ±50 %/jour
    active = N.sum(axis=1) > 0
    if not active.any():
        return pd.Series(dtype=float)
    return r.loc[active.idxmax(): active[::-1].idxmax()]

def stats(r):
    nav = (1 + r).cumprod(); T = len(r)
    cagr = nav.iloc[-1] ** (252 / T) - 1 if nav.iloc[-1] > 0 else np.nan
    return dict(cagr=cagr, vol=r.std() * np.sqrt(252),
                sharpe=r.mean() / r.std() * np.sqrt(252) if r.std() > 0 else np.nan,
                maxdd=(nav / nav.cummax() - 1).min())

In [9]:
rm = daily_return(Nm)
print(f"{DEMO} :", {k: round(v, 3) for k, v in stats(rm).items()})

John Fetterman : {'cagr': np.float64(0.797), 'vol': np.float64(0.21), 'sharpe': np.float64(2.904), 'maxdd': np.float64(-0.162)}


## 3. Excès, Information Ratio & ajustement par le marché ($\beta$)
Marché = SPY. Série d'excès et **IR** (suppose $\beta=1$) :
$$x(t)=r_P(t)-r_{SPY}(t),\quad \mathrm{IR}=\frac{\overline{x}}{\sigma(x)}\sqrt{252},\quad t=\mathrm{IR}\sqrt{n/252}>1{,}645$$
**Ajustement par le marché.** On régresse $r_P = \alpha + \beta\,r_{SPY} + \varepsilon$ :
$$\beta=\frac{\mathrm{Cov}(r_P,r_{SPY})}{\mathrm{Var}(r_{SPY})},\quad \alpha=\overline{r_P}-\beta\,\overline{r_{SPY}},\quad \text{appraisal}=\frac{\alpha}{\sigma(\varepsilon)}\sqrt{252}$$
- $\beta$ = levier de style ; $\alpha$ = talent ; l'**appraisal** = l'IR du résidu (talent risque-ajusté, marché retiré).
- $\beta=1$ : appraisal = IR ; sinon l'IR mélange exposition et talent.

In [10]:
def info_ratio(r):
    x = (r - r_spy.reindex(r.index)).dropna()
    return x.mean() / x.std() * np.sqrt(252) if x.std() > 0 else np.nan

def alpha_beta(r):
    """Régression r = alpha + beta*r_spy + eps -> beta, alpha annualisé, ratio d'appréciation."""
    d = pd.concat([r.rename("y"), r_spy.reindex(r.index).rename("x")], axis=1).dropna()
    if len(d) < 30 or d["x"].var() == 0:
        return dict(beta=np.nan, alpha_ann=np.nan, appraisal=np.nan)
    beta, alpha = np.polyfit(d["x"].values, d["y"].values, 1)      # pente, ordonnée
    eps = d["y"].values - (alpha + beta * d["x"].values)
    se = eps.std(ddof=2)
    return dict(beta=beta, alpha_ann=(1 + alpha) ** 252 - 1,
                appraisal=(alpha / se * np.sqrt(252)) if se > 0 else np.nan)

In [11]:
ab = alpha_beta(rm)
print(f"{DEMO} : IR = {info_ratio(rm):.2f} | beta = {ab['beta']:.2f} | alpha annualisé = {ab['alpha_ann']:+.1%} | appraisal = {ab['appraisal']:.2f}")

John Fetterman : IR = 2.48 | beta = 1.17 | alpha annualisé = +40.2% | appraisal = 2.24


### Préparation : la série de rendement de chaque membre (calculée une fois)
`member_shares`/`daily_return` sont cumulatifs $\Rightarrow$ tronquer à une date ne regarde jamais le futur.

In [12]:
wf_ret, wf_traded = {}, {}
for bid, g in df.groupby("bioguide_id"):
    N = member_shares(g)
    if N is None:
        continue
    r = daily_return(N)
    if len(r) < 2 or r.std() == 0:
        continue
    wf_ret[bid] = r
    wf_traded[bid] = pd.to_datetime(np.sort(g["traded"].values))
print(len(wf_ret), "membres avec une série de rendement")

266 membres avec une série de rendement


## 4. Éligibilité, tests multiples & classement IR vs appraisal
- **Éligibilité** : $n_{\text{trades}}\ge 10$ et $n_{\text{jours}}\ge 126$ (~6 mois).
- **Tests multiples** : sur $M$ membres à 5 %, on attend $0{,}05\,M$ significatifs par hasard ; Bonferroni durcit à $t\gtrsim 3{,}5$ pour $M\simeq 223$.
- On classe par **IR** (actuel) ET par **appraisal** (talent β-propre) pour voir qui monte/descend une fois le beta retiré.

In [13]:
MIN_TRADES, MIN_DAYS = 10, 126
rows = []
for bid, r in wf_ret.items():
    ab = alpha_beta(r)
    rows.append(dict(name=df.loc[df.bioguide_id == bid, "name"].iloc[0],
                     n_trades=len(wf_traded[bid]), n_days=len(r), IR=info_ratio(r),
                     beta=ab["beta"], alpha_ann=ab["alpha_ann"], appraisal=ab["appraisal"]))
tab = pd.DataFrame(rows).sort_values("IR", ascending=False).reset_index(drop=True)
elig = tab[(tab.n_trades >= MIN_TRADES) & (tab.n_days >= MIN_DAYS)].copy()
elig["t"] = elig.IR * np.sqrt(elig.n_days / 252)
sig = elig[elig.t > 1.645]
M = len(elig)
print(f"{len(tab)} classés -> {M} éligibles -> {len(sig)} significatifs (t>1,645), ~{0.05*M:.0f} attendus par hasard ; Bonferroni t>=3,5")

266 classés -> 223 éligibles -> 20 significatifs (t>1,645), ~11 attendus par hasard ; Bonferroni t>=3,5


In [14]:
cols = ["name", "n_trades", "IR", "beta", "alpha_ann", "appraisal"]
print("=== TOP 10 par IR (classement actuel, beta=1) ===")
print(elig.sort_values("IR", ascending=False).head(10)[cols].round(3).to_string(index=False))
print("\n=== TOP 10 par APPRAISAL (talent beta-propre) ===")
print(elig.sort_values("appraisal", ascending=False).head(10)[cols].round(3).to_string(index=False))

=== TOP 10 par IR (classement actuel, beta=1) ===
               name  n_trades    IR  beta  alpha_ann  appraisal
     John Fetterman        10 2.482 1.170      0.402      2.242
       Ashley Moody        23 1.670 1.792      0.480      1.365
    Morgan McGarvey        24 1.502 1.534      0.240      1.081
       Lisa McClain      1479 1.264 1.759      0.233      0.835
    Bruce Westerman       208 1.194 0.948      0.131      1.309
        Kim Schrier        40 1.106 1.220      0.188      1.075
  Daniel S Sullivan       100 0.999 1.572      0.075      0.432
      Thomas Suozzi       662 0.902 1.222      0.051      0.589
Barbara J. Comstock        85 0.899 1.201      0.075      0.659
          Tim Moore       163 0.822 1.169      0.126      0.655

=== TOP 10 par APPRAISAL (talent beta-propre) ===
           name  n_trades    IR  beta  alpha_ann  appraisal
 John Fetterman        10 2.482 1.170      0.402      2.242
   Ashley Moody        23 1.670 1.792      0.480      1.365
Bruce Westerman

In [15]:
# Qui monte / descend le plus quand on passe de l'IR à l'appraisal ?
e = elig.copy()
e["rang_IR"] = e.IR.rank(ascending=False)
e["rang_appr"] = e.appraisal.rank(ascending=False)
e["delta_rang"] = e.rang_IR - e.rang_appr      # >0 = MONTE en passant à l'appraisal
top_reshuffle = e[e.n_trades >= MIN_TRADES].sort_values("delta_rang")
print("PLUS GROSSE CHUTE (beta gonflait leur IR) :")
print(top_reshuffle.head(5)[["name", "IR", "beta", "appraisal", "rang_IR", "rang_appr"]].round(2).to_string(index=False))
print("\nPLUS GROSSE MONTÉE (vrai talent masqué par l'IR) :")
print(top_reshuffle.tail(5)[["name", "IR", "beta", "appraisal", "rang_IR", "rang_appr"]].round(2).to_string(index=False))

PLUS GROSSE CHUTE (beta gonflait leur IR) :
                 name    IR  beta  appraisal  rang_IR  rang_appr
          Cleo Fields  0.65  1.56      -0.27     22.0      168.0
        Shri Thanedar  0.04  2.47      -0.32     80.0      177.0
Kelly Louise Morrison -0.18  1.52      -0.60    117.0      208.0
   William R. Timmons -0.12  1.40      -0.43    107.0      197.0
          James Comer -0.22  1.18      -0.54    126.0      205.0

PLUS GROSSE MONTÉE (vrai talent masqué par l'IR) :
                     name    IR  beta  appraisal  rang_IR  rang_appr
         Randy Neugebauer -0.29  0.36       0.23    142.0       52.0
         William L. Owens -0.60  0.32       0.03    199.0       93.0
     Charlie Joseph Crist -0.58  0.27       0.07    197.0       84.0
Richard Dean Dr McCormick -0.86  0.58       0.12    215.0       71.0
           Lindsey Graham -1.03  0.09       0.52    220.0       25.0


## 5. Classement walk-forward (point-in-time)
À chaque coupe fin d'année $Y$, on ne garde que l'info $\le 31/12/Y$ :
$$\mathrm{classement}(Y)=\{\text{éligibles}\wedge t>1{,}645\}\ \text{triés par IR}.$$
`select_topK(Y,K)` prend les $K$ premiers (repli si $<K$). Aucun regard vers le futur.

In [16]:
def classement_a_fin(Y, min_trades=MIN_TRADES, min_days=MIN_DAYS, seuil_t=1.645):
    yend = pd.Timestamp(f"{Y}-12-31")
    rows = []
    for bid, r in wf_ret.items():
        rr = r.loc[:yend]
        n_trades = int((wf_traded[bid] <= yend).sum())
        n_days = len(rr)
        if n_days < 2 or rr.std() == 0:
            continue
        if n_trades < min_trades or n_days < min_days:
            continue
        x = (rr - r_spy.reindex(rr.index)).dropna()
        if len(x) < 2 or x.std() == 0:
            continue
        ir = x.mean() / x.std() * np.sqrt(252)
        t = ir * np.sqrt(n_days / 252)
        if t <= seuil_t:
            continue
        rows.append(dict(bioguide_id=bid, name=df.loc[df.bioguide_id == bid, "name"].iloc[0],
                         n_trades=n_trades, n_days=n_days, IR=ir, t=t))
    cols = ["bioguide_id", "name", "n_trades", "n_days", "IR", "t"]
    return (pd.DataFrame(rows).sort_values("IR", ascending=False).reset_index(drop=True)
            if rows else pd.DataFrame(columns=cols))

def select_topK(Y, K):
    c = classement_a_fin(Y)
    return list(zip(c["IR"].head(K), c["bioguide_id"].head(K)))

In [17]:
resume = pd.DataFrame([
    dict(fenetre=f"2014-{Y}", n_significatifs=len(c),
         top1=c.name.iloc[0] if len(c) else "-",
         top2=c.name.iloc[1] if len(c) > 1 else "-")
    for Y in range(2014, 2027) for c in [classement_a_fin(Y)]])
resume

,fenetre,n_significatifs,top1,top2
0,2014-2014,1,John F Reed,-
1,2014-2015,4,David B. McKinley,Susan A. Davis
2,2014-2016,2,Billy Long,"Angus S King, Jr."
3,2014-2017,4,Barbara J. Comstock,Billy Long
4,2014-2018,5,Denny Heck,Dwight Evans
5,2014-2019,8,Dwight Evans,Denny Heck
6,2014-2020,19,Donald Sternoff Beyer,Gilbert Cisneros
7,2014-2021,13,Barbara J. Comstock,MARK R WARNER
8,2014-2022,7,Suzan K. DelBene,Barbara J. Comstock
9,2014-2023,12,Greg Landsman,Daniel S Sullivan


## 6. Stratégie top-$K$ : la bonne combinaison
Sélection fin $Y$, on suit le panier en $Y{+}1$. Poids initiaux $w_k=1/K$ ou $\mathrm{IR}_k/\sum_j \mathrm{IR}_j$.

**Écueil** — la moyenne à poids figés $\sum_k w_k r_{m_k}$ (avec $w_k$ constants) revient à **rebalancer chaque jour** ; « copier et **tenir** » laisse les poids **dériver**.

**Combinaison correcte (buy-and-hold)** : $\mathrm{NAV}_k(t)=\prod_{s\le t}(1+r_{m_k}(s))$, $N(t)=\sum_k w_k\,\mathrm{NAV}_k(t)$, puis
$$r_{\text{strat}}(t)=\frac{N(t)}{N(t-1)}-1,\qquad w_k(t)=\frac{w_k\,\mathrm{NAV}_k(t-1)}{\sum_j w_j\,\mathrm{NAV}_j(t-1)}.$$

In [18]:
def run_strategy(K, mode, years=range(2015, 2026), combine="drift"):
    parts = []
    for Y in years:
        sel = select_topK(Y, K)
        if not sel:
            continue
        if mode == "equal":
            w = np.ones(len(sel)) / len(sel)
        else:
            irs = np.array([ir for ir, _ in sel])
            w = irs / irs.sum() if irs.sum() > 0 else np.ones(len(sel)) / len(sel)
        idx = cal[cal.year == (Y + 1)]
        if len(idx) == 0:
            continue
        R = pd.DataFrame({bid: wf_ret[bid].reindex(idx).fillna(0.0) for _, bid in sel})
        if combine == "fixed":                       # ANCIEN : poids figés = rebalancement quotidien
            parts.append((R * w).sum(axis=1))
        else:                                         # DRIFT (buy-and-hold) : poids qui dérivent
            navk = (1 + R).cumprod()                  # NAV_k normalisée à 1 en début d'année
            navc = (navk * w).sum(axis=1)             # N(t) = Σ w_k NAV_k(t)
            ry = navc / navc.shift(1) - 1
            ry.iloc[0] = navc.iloc[0] - 1             # 1er jour : NAV 1 -> navc[0]
            parts.append(ry)
    return pd.concat(parts).sort_index()

In [19]:
# Exemple jouet (A=Apple, B=Coca, w=1/2,1/2) : figé (faux) vs dérivé (correct)
R_toy = pd.DataFrame({"A": [1.00, 0.00], "B": [0.00, 0.20]}, index=["jour1", "jour2"])
w = np.array([0.5, 0.5])
fixed = (R_toy * w).sum(axis=1)
navk = (1 + R_toy).cumprod(); navc = (navk * w).sum(axis=1)
drift = navc / navc.shift(1) - 1; drift.iloc[0] = navc.iloc[0] - 1
print("figé   :", {k: f"{v:+.2%}" for k, v in fixed.items()})
print("dérivé :", {k: f"{v:+.2%}" for k, v in drift.items()}, " <- jour2 = 6,67 % (correct)")

figé   : {'jour1': '+50.00%', 'jour2': '+10.00%'}
dérivé : {'jour1': '+50.00%', 'jour2': '+6.67%'}  <- jour2 = 6,67 % (correct)


## 7. Évaluation annuelle & ajustement au marché
Excès composé par année ($G_Y(r)=\prod_{t\in Y}(1+r)$) : $x_{\text{strat},Y}=G_Y(r_{\text{strat}})-G_Y(r_{SPY})$,
$\mathrm{IR}_{\text{ann}}=\overline{x}/\sigma(x)$, $t=\mathrm{IR}_{\text{ann}}\sqrt{n}$ (seuil ~1,81 à $n=11$).

**Ajustement $\beta$ de la stratégie** : même régression $r_{\text{strat}}=\alpha+\beta\,r_{SPY}+\varepsilon$.

In [20]:
def yearly_excess(r):
    b = r_spy.reindex(r.index).fillna(0.0)
    return pd.Series({y: ((1 + r[r.index.year == y]).prod() - 1) - ((1 + b[r.index.year == y]).prod() - 1)
                      for y in sorted(set(r.index.year))})

def eval_yearly(r):
    e = yearly_excess(r); n = len(e); ir_an = e.mean() / e.std(ddof=1)
    return dict(exces_moyen=e.mean(), IR_annuel=ir_an, t=ir_an * np.sqrt(n), gagnantes=f"{int((e > 0).sum())}/{n}")

# Combinaison CORRIGÉE (drift) — comparaison avec l'ancienne (figée) pour mesurer l'écart
comp = pd.DataFrame({
    "1/K figé (ancien)":    eval_yearly(run_strategy(4, "equal", combine="fixed")),
    "1/K dérive (corrigé)": eval_yearly(run_strategy(4, "equal", combine="drift")),
    "IR  figé (ancien)":    eval_yearly(run_strategy(4, "ir", combine="fixed")),
    "IR  dérive (corrigé)": eval_yearly(run_strategy(4, "ir", combine="drift")),
}).T
comp.round(3)

,exces_moyen,IR_annuel,t,gagnantes
1/K figé (ancien),0.036491,0.344214,1.141628,7/11
1/K dérive (corrigé),0.031021,0.284383,0.943191,7/11
IR figé (ancien),0.035121,0.327474,1.086108,7/11
IR dérive (corrigé),0.029944,0.271288,0.899762,7/11


In [21]:
# Ajustement beta AU NIVEAU STRATÉGIE (sur la combinaison corrigée)
for mode in ["equal", "ir"]:
    r = run_strategy(4, mode, combine="drift")
    ab = alpha_beta(r)
    print(f"strat top-4 [{mode:5s}] : beta = {ab['beta']:.2f} | alpha annualisé = {ab['alpha_ann']:+.2%} | appraisal = {ab['appraisal']:.2f}")

strat top-4 [equal] : beta = 1.20 | alpha annualisé = -0.44% | appraisal = -0.04


strat top-4 [ir   ] : beta = 1.21 | alpha annualisé = -0.64% | appraisal = -0.06


In [22]:
print("=" * 60)
print("RÉSUMÉ — copy-trading du Congrès vs SPY (combinaison corrigée)")
print("=" * 60)
print(f"Membres avec historique        : {len(wf_ret)}")
print(f"Éligibles (>=10 trades, >=126j) : {len(elig)}")
print(f"Significatifs (t > 1,645)       : {len(sig)}")
for mode in ["equal", "ir"]:
    m = eval_yearly(run_strategy(4, mode, combine="drift"))
    print(f"Strat top-4 [{mode:5s}] : excès {m['exces_moyen']*100:+.1f}%/an | IR annuel {m['IR_annuel']:.2f} | t {m['t']:.2f} | gagnantes {m['gagnantes']}")
print("=" * 60)

RÉSUMÉ — copy-trading du Congrès vs SPY (combinaison corrigée)
Membres avec historique        : 266
Éligibles (>=10 trades, >=126j) : 223
Significatifs (t > 1,645)       : 20


Strat top-4 [equal] : excès +3.1%/an | IR annuel 0.28 | t 0.94 | gagnantes 7/11


Strat top-4 [ir   ] : excès +3.0%/an | IR annuel 0.27 | t 0.90 | gagnantes 7/11
